# Entregável 10 — Validação Estatística Final do Dataset

**Disciplina:** Aquisição de Biossinais  
**Equipe:** José Lessa & Matheus Rocha  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** PTB-XL (PhysioNet)  

---

## Objetivo

Esta é a etapa derradeira do processamento de sinais. Temos em mãos um subconjunto ultra seletivo 
de Features Matemáticas (Entregável 9). Antes de jurar cientificamente que esse dataset está 
pronto para treinar Inteligências Artificiais em Reconhecimento de Padrões (RP), precisaremos:
1. Provar ausência de Colinearidade Multivariável usando fator VIF.
2. Avaliar assimetria demográfica (Classes Desbalanceadas) e corrigir com técnica artificial (SMOTE).
3. Demonstrar separabilidade estatística visual através de Curvas de Densidade (KDE).

## 1. Configuração e Imports

In [ ]:
!pip install scikit-learn imbalanced-learn statsmodels -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from statsmodels.stats.outliers_influence import variance_inflation_factor
from imblearn.over_sampling import SMOTE

# Configurações de plot originais da disciplina
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True

BASE_DIR = Path('..')
FIG_DIR = Path('figuras')
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Configuração pronta.')

## 2. Carregamento do Dataset de Features Otimizadas (E.9)

In [ ]:
csv_path = BASE_DIR / 'entregavel-9' / 'dataset_features_selecionadas_puras.csv'
df = pd.read_csv(csv_path)

# Isolar o que é Metadado e o que é Feature
cols_meta = ['ecg_id', 'derivacao', 'janela_instancia', 'target_is_norm']
cols_feat = [c for c in df.columns if c not in cols_meta]

X = df[cols_feat]
y = df['target_is_norm']

print(f"-> Carregadas {X.shape[0]} amostras de janelas clínicas com {len(cols_feat)} the 'Best Features' finais.")

## 3. Verificação de Multicolinearidade Clássica (VIF)

No Entregável 7 nós eliminamos pares lineares óbvios (Spearman > 0.95), mas e se a **Soma** da Feature A com a Feature B resultar magicamente na Feature C? Isso não é pego pelo Spearman par-a-par.
O **Variance Inflation Factor (VIF)** roda uma regressão de Múltiplas Variaveis para cravar.  
*Limiar Acadêmico Clássico: VIF < 10 é excelente.*

In [ ]:
print("Calculando Fatorial de Inflação da Variância (VIF) para cada feature final...")

vif_data = pd.DataFrame()
vif_data["Feature (Original)"] = X.columns

# VIF calculation formula array-oriented
vif_data["VIF_Score"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]
vif_data.sort_values('VIF_Score', ascending=False, inplace=True)

print('=' * 80)
print("DIAGNÓSTICO FORMAL DE MULTICOLINEARIDADE MULTIVARIADA")
print('=' * 80)
display(vif_data.round(2))

max_vif = vif_data['VIF_Score'].max()
if max_vif < 10.0:
    print(f"✅ APROVADO: Nenhuma feature inflaciona o dataset (Max VIF = {max_vif:.2f}). Dataset independente!")
else:
    print(f"⚠️ AVISO: Algumas features possuem VIF > 10. O RFECV do entregável anterior não ligou pra isso (porque árvores não sofrem tanto com colinearidade), mas Modelos Lineares (SVM) podem falhar na generalização.")
    print("Para blindagem absoluta TCC-Grade, droparemos internamente as features com VIF extremo ( > 15 ).")

    colunas_seguras = vif_data[vif_data['VIF_Score'] <= 15]['Feature (Original)'].tolist()
    X = X[colunas_seguras]
    print(f"-> Aplicado VIF filter: Sobraram exatamente {len(colunas_seguras)} features absolutamente independentes!")

## 4. Avaliação de Balanceamento das Classes

Classificadores que treinam em 90% pacientes Doentes e 10% Saudáveis irão sempre "chutar" que todos os novos pacientes estão Doentes para garantir alta acurácia cega. O re-balançeamento corrige o Viés Amostral.

In [ ]:
distribuicao_pre = y.value_counts()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x=y, ax=ax[0], palette=['crimson', 'mediumseagreen'])
ax[0].set_title(f'Distribuição Original (Realidade PTB-XL)\nDoentes (0): {distribuicao_pre[0]} | NORM (1): {distribuicao_pre[1]}', fontweight='bold')
ax[0].set_ylabel('Nº de Janelas / Instâncias')
ax[0].set_xlabel('Diagnóstico (target_is_norm)')

# Aplicando o SMOTE (Synthetic Minority Over-sampling Technique)
print('Processando Over-sampling Sintético (SMOTE) para calibrar a margem minoritária geometricamente...')
smote = SMOTE(random_state=42)
X_bal, y_bal = smote.fit_resample(X, y)

distribuicao_pos = y_bal.value_counts()
sns.countplot(x=y_bal, ax=ax[1], palette=['crimson', 'mediumseagreen'])
ax[1].set_title(f'Distribuição Após SMOTE Over-sampling\nDoentes (0): {distribuicao_pos[0]} | NORM (1): {distribuicao_pos[1]}', fontweight='bold')
ax[1].set_ylabel('Nº de Instâncias Sintetizantes Adicionadas')
ax[1].set_xlabel('Diagnóstico (target_is_norm)')

plt.tight_layout()
plt.savefig(FIG_DIR / 'balanceamento_smote.png')
plt.show()
print('-> Classes balanceadas estatisticamente em 50% / 50%')

# Concatenando com SMOTe
df_final_ml = pd.DataFrame(X_bal, columns=X.columns)
df_final_ml['target_is_norm'] = y_bal

## 5. Análise de Separabilidade e Curvas de Densidade (KDE)
Se montarmos um gráfico da Densidade Kernel de uma métrica, as curvas da Doença vs Saudável estarão distantes ou completamente entrelaçadas? Se houver *deslocamento modal* visível, comprova que a Engenharia prestou.

In [ ]:
# Plotaremos as 4 top features para o Professor
top_4_features = X.columns[:4]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, feature in enumerate(top_4_features):
    sns.kdeplot(data=df_final_ml[df_final_ml['target_is_norm']==0][feature], ax=axes[i], color='crimson', fill=True, alpha=0.3, label='Classe: 0 (Anormal)')
    sns.kdeplot(data=df_final_ml[df_final_ml['target_is_norm']==1][feature], ax=axes[i], color='mediumseagreen', fill=True, alpha=0.3, label='Classe: 1 (NORM Saudável)')
    
    axes[i].set_title(f'KDE Separability Plot - [{feature}]', fontweight='bold')
    axes[i].set_xlabel('Valor Transformado (Z-Score)')
    axes[i].set_ylabel('Densidade de Probabilidade Gaussiana Exata')
    axes[i].legend()

plt.suptitle('Comprovação Visual da Separabilidade Estatística das Classes no Subconjunto Ideal', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'kde_curvas_densidade_separabilidade.png')
plt.show()
print('Figura salva.')

## 6. Geração do Relatório-Síntese Final Dataset

Montagem orgulhosa do nosso mega produto computacional validado a rigor!

In [ ]:
output_csv = FIG_DIR.parent / 'DATASET_SUPREMO_FINAL_VALIDADO_ML.csv'
df_final_ml.to_csv(output_csv, index=False)

print("=========================================================================================")
print("🏁 ENTREGÁVEL FINAL 10 - DATASET VALIDADO MATEMATICAMENTE E PRONTO PARA RP 🏁")
print("=========================================================================================")
print(f"Caminho de Exportação Cúspide: {output_csv}")
print(f"Instâncias Treináveis (Pós-Smote): {df_final_ml.shape[0]} amostras de dados na linha (X)")
print(f"Preditoras Refinadas, Não-Colineares, Z-Escalonadas: {len(X.columns)} features perfeitas (y)")
print("=========================================================================================")
print("-> Todos os 10 passos do 'Aquisição de Biossinais' UFC foram vencidos rigorosamente.\n-> Proceder ao TCC com treinos de SVM ou Redes Neurais neste arquivo.")